# Portfolio of Evidence: Medical-Aid Review Text Analysis

## Business requirement
This notebook identifies **customer sentiment** and the **main areas of concern** in medical-aid reviews. It uses:

- **TF-IDF + Logistic Regression** for negative, neutral and positive sentiment classification.
- **Latent Dirichlet Allocation (LDA)** for unsupervised topic discovery.
- Integrated topic/sentiment prioritisation for client recommendations.

> **Academic transparency:** `medical_aid_customer_reviews.csv` is a synthetic, anonymised teaching dataset. It contains no scraped HelloPeter reviews or real customer records. The methodology is suitable for a POE, but the generated findings must not be presented as evidence about a real provider.

## 1. Model and dataset justification

### Model choice
Logistic Regression is an interpretable and reproducible multiclass sentiment classifier. TF-IDF gives greater weight to informative words and phrases. LDA complements sentiment analysis by identifying recurring themes without requiring topic labels.

### Dataset suitability
The dataset contains 720 review records, review text, date, rating, sentiment, channel, province, provider and a known concern category. Text supports both models; sentiment supports supervised evaluation; dates and ratings support EDA and reasonableness checks.

### Quality and limitations
The analysis checks missingness, duplicates, invalid ratings, class balance and review length. Synthetic language is cleaner than real review-platform text. Real data may contain selection bias, sarcasm, multilingual content, spelling errors, spam and personally identifiable information.

## 2. Concise analysis plan

| Stage | Activity | Purpose |
|---|---|---|
| Audit | Check schema, missingness, duplicates and invalid values | Validate quality |
| EDA | Analyse sentiment, ratings, dates, length, providers and phrases | Inform modelling |
| Prepare | Clean text while retaining negation | Preserve sentiment meaning |
| Features | TF-IDF unigrams/bigrams; count vectors for LDA | Create model inputs |
| Train | Stratified split, Logistic Regression and LDA | Address both client needs |
| Tune | 5-fold grid search and candidate topic counts | Improve robustness |
| Evaluate | Accuracy, macro F1, per-class metrics, confusion matrix and topic inspection | Assess effectiveness |
| Report | Integrate sentiment and topic volume | Prioritise concerns |

In [ ]:
from pathlib import Path
import re, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score
from sklearn.decomposition import LatentDirichletAllocation

RANDOM_STATE = 42
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_colwidth', 120)

paths=[Path('medical_aid_customer_reviews.csv'),Path('/mnt/data/medical_aid_customer_reviews.csv')]
DATA_PATH=next((x for x in paths if x.exists()),None)
if DATA_PATH is None: raise FileNotFoundError('Place the CSV in the notebook folder.')
df=pd.read_csv(DATA_PATH,parse_dates=['review_date'])
print(f'Rows: {len(df):,}; columns: {df.shape[1]}')
df.head()

## 3. Data audit and cleaning
Cleaning is deliberately conservative. Negation words such as **not**, **no** and **nor** are retained because removing them can reverse meaning.

In [ ]:
quality=pd.DataFrame({'missing':df.isna().sum(),'missing_%':(100*df.isna().mean()).round(2),'unique':df.nunique()})
display(quality)
print('Exact duplicates:',df.duplicated().sum())
print('Duplicate IDs:',df['review_id'].duplicated().sum())
print('Blank reviews:',df['review_text'].fillna('').str.strip().eq('').sum())

def clean_text(x):
    x=str(x).lower()
    x=re.sub(r'https?://\S+|www\.\S+',' ',x)
    x=re.sub(r'<[^>]+>',' ',x)
    x=re.sub(r"[^a-z\s']",' ',x)
    return re.sub(r'\s+',' ',x).strip()

before=len(df)
df=df.drop_duplicates('review_id')
df=df[df.review_text.notna() & df.review_text.str.strip().ne('') & df.rating.between(1,5)].copy()
df['clean_text']=df.review_text.map(clean_text)
df['word_count']=df.clean_text.str.split().str.len()
df['char_count']=df.clean_text.str.len()
print('Rows removed:',before-len(df))
display(df[['review_text','clean_text','word_count']].head())

## 4. Exploratory Data Analysis
The EDA is tied to modelling. Class imbalance motivates stratification and balanced class weights. Phrase frequencies motivate bigrams. Review length identifies weak documents. Time and provider charts expose distributional patterns.

In [ ]:
# Graphs 1-2: sentiment and rating distributions
fig,axes=plt.subplots(1,2,figsize=(14,5))
order=['negative','neutral','positive']
sc=df.sentiment.value_counts().reindex(order)
axes[0].bar(sc.index,sc.values,color=['#C0392B','#F4B942','#2E8B57']); axes[0].set_title('Sentiment Distribution'); axes[0].set_ylabel('Reviews')
for i,v in enumerate(sc): axes[0].text(i,v+3,str(v),ha='center')
rc=df.rating.value_counts().sort_index()
axes[1].bar(rc.index.astype(str),rc.values,color='#355C7D'); axes[1].set_title('Rating Distribution'); axes[1].set_xlabel('Rating'); axes[1].set_ylabel('Reviews')
plt.tight_layout(); plt.show()

In [ ]:
# Graphs 3-4: review length and monthly volume
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].boxplot([df.loc[df.sentiment.eq(s),'word_count'] for s in order],labels=[s.title() for s in order],showfliers=False)
axes[0].set_title('Review Length by Sentiment'); axes[0].set_ylabel('Word Count')
monthly=df.set_index('review_date').resample('MS').size()
axes[1].plot(monthly.index,monthly.values,marker='o',color='#6C5B7B'); axes[1].set_title('Monthly Review Volume'); axes[1].set_ylabel('Reviews')
plt.tight_layout(); plt.show()
display(df.groupby('sentiment').word_count.describe().round(2))

In [ ]:
# Graph 5: provider sentiment mix
provider_sent=pd.crosstab(df.provider,df.sentiment,normalize='index')[order]*100
provider_sent.plot(kind='bar',stacked=True,color=['#C0392B','#F4B942','#2E8B57'])
plt.title('Sentiment Mix by Provider'); plt.ylabel('Percentage'); plt.xlabel('Provider')
plt.legend(title='Sentiment',bbox_to_anchor=(1.02,1)); plt.xticks(rotation=15,ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Graphs 6-7: frequent unigrams and bigrams
stop=list(set(ENGLISH_STOP_WORDS)-{'no','not','nor'})
def top_terms(ngram):
    v=CountVectorizer(stop_words=stop,ngram_range=ngram,min_df=2)
    m=v.fit_transform(df.clean_text); c=np.asarray(m.sum(axis=0)).ravel(); t=np.array(v.get_feature_names_out()); ix=c.argsort()[-15:]
    return t[ix],c[ix]
fig,axes=plt.subplots(1,2,figsize=(15,6))
for ax,ng,title,col in zip(axes,[(1,1),(2,2)],['Top 15 Unigrams','Top 15 Bigrams'],['#4C78A8','#F58518']):
    t,c=top_terms(ng); ax.barh(t,c,color=col); ax.set_title(title); ax.set_xlabel('Frequency')
plt.tight_layout(); plt.show()

## 5. Sentiment feature selection and training
Ratings and known concern categories are excluded from predictors to prevent leakage. TF-IDF learns only from training folds inside the Pipeline. Five-fold stratified grid search compares regularisation and n-gram settings using **macro F1**.

In [ ]:
X=df.clean_text; y=df.sentiment
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)
pipe=Pipeline([('tfidf',TfidfVectorizer(stop_words=stop,min_df=2,max_df=.95,sublinear_tf=True)),('clf',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE))])
params={'tfidf__ngram_range':[(1,1),(1,2)],'tfidf__max_features':[3000,6000],'clf__C':[.1,1,10]}
cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE)
grid=GridSearchCV(pipe,params,scoring='f1_macro',cv=cv,n_jobs=-1,return_train_score=True)
grid.fit(X_train,y_train)
print('Best parameters:',grid.best_params_)
print('Best CV macro F1:',round(grid.best_score_,4))

## 6. Sentiment evaluation
Accuracy is included, but macro F1 and per-class recall are more informative when classes are imbalanced. The confusion matrix identifies which sentiments are confused.

In [ ]:
best_model=grid.best_estimator_; pred=best_model.predict(X_test)
print('Test accuracy:',round(accuracy_score(y_test,pred),4))
print(classification_report(y_test,pred,digits=4))
ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=order,display_labels=[x.title() for x in order],cmap='Blues')
plt.title('Sentiment Confusion Matrix'); plt.tight_layout(); plt.show()

In [ ]:
# Graphs 8-9: tuning results and influential features
res=pd.DataFrame(grid.cv_results_)
res['label']=res.params.apply(lambda p:f"C={p['clf__C']}, ngram={p['tfidf__ngram_range']}, max={p['tfidf__max_features']}")
top=res.nlargest(10,'mean_test_score').sort_values('mean_test_score')
plt.barh(top.label,top.mean_test_score,color='#54A24B'); plt.title('Top Configurations by CV Macro F1'); plt.xlabel('Mean CV Macro F1'); plt.tight_layout(); plt.show()

vec=best_model.named_steps['tfidf']; clf=best_model.named_steps['clf']; names=np.array(vec.get_feature_names_out())
fig,axes=plt.subplots(1,len(clf.classes_),figsize=(18,5))
for ax,cl,coef in zip(axes,clf.classes_,clf.coef_):
    ix=np.argsort(coef)[-12:]; ax.barh(names[ix],coef[ix]); ax.set_title(f'Terms Predicting {cl.title()}')
plt.tight_layout(); plt.show()

## 7. LDA topic modelling
LDA uses count-based features. Candidate topic counts are compared using held-out perplexity. Topic selection must also consider distinctiveness, representative reviews and stakeholder interpretability.

In [ ]:
tr,va=train_test_split(df.clean_text,test_size=.20,random_state=RANDOM_STATE)
cvect=CountVectorizer(stop_words=stop,min_df=4,max_df=.90,ngram_range=(1,2),max_features=5000)
Ct=cvect.fit_transform(tr); Cv=cvect.transform(va)
rows=[]; models={}
for k in [4,5,6,7,8,9]:
    m=LatentDirichletAllocation(n_components=k,max_iter=30,learning_method='batch',random_state=RANDOM_STATE).fit(Ct)
    models[k]=m; rows.append([k,m.perplexity(Ct),m.perplexity(Cv)])
lda_eval=pd.DataFrame(rows,columns=['topics','train_perplexity','validation_perplexity'])
display(lda_eval.round(2))
plt.plot(lda_eval.topics,lda_eval.validation_perplexity,marker='o',label='Validation')
plt.plot(lda_eval.topics,lda_eval.train_perplexity,marker='o',label='Training')
plt.title('LDA Topic-Count Comparison'); plt.xlabel('Topics'); plt.ylabel('Perplexity (lower is better)'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Six topics reflect the six teaching concern areas; real projects should also use coherence and human review.
lda=models[6]; terms=np.array(cvect.get_feature_names_out())
fig,axes=plt.subplots(2,3,figsize=(18,10))
summary=[]
for i,ax in enumerate(axes.flat):
    ix=lda.components_[i].argsort()[-10:]
    ax.barh(terms[ix],lda.components_[i][ix],color='#B279A2'); ax.set_title(f'Topic {i+1}')
    summary.append([i+1,', '.join(terms[ix][::-1])])
plt.suptitle('Top Terms in Six LDA Topics',fontweight='bold'); plt.tight_layout(); plt.show()
display(pd.DataFrame(summary,columns=['topic','top_terms']))

In [ ]:
# Representative reviews and dominant topic
Call=cvect.transform(df.clean_text); probs=lda.transform(Call)
df['dominant_topic']=probs.argmax(1)+1; df['topic_confidence']=probs.max(1)
representative=(df.sort_values('topic_confidence',ascending=False).groupby('dominant_topic').head(3)[['dominant_topic','topic_confidence','review_text','known_concern_category']])
display(representative.sort_values('dominant_topic'))

## 8. Integrated topic and sentiment findings
Concern priority combines volume and negative sentiment. Topic labels should be assigned after reading the top terms and representative reviews.

In [ ]:
# Graphs 11-13: topic volume, sentiment composition and priority map
fig,axes=plt.subplots(1,2,figsize=(15,5))
vol=df.dominant_topic.value_counts().sort_index(); axes[0].bar(vol.index.astype(str),vol.values,color='#4C78A8'); axes[0].set_title('Review Volume by LDA Topic'); axes[0].set_xlabel('Topic')
ts=pd.crosstab(df.dominant_topic,df.sentiment,normalize='index')[order]*100
ts.plot(kind='bar',stacked=True,ax=axes[1],color=['#C0392B','#F4B942','#2E8B57']); axes[1].set_title('Sentiment by LDA Topic'); axes[1].set_ylabel('Percentage')
plt.tight_layout(); plt.show()

priority=pd.DataFrame({'volume':df.groupby('dominant_topic').size(),'negative_rate':100*df.assign(neg=df.sentiment.eq('negative')).groupby('dominant_topic').neg.mean(),'confidence':df.groupby('dominant_topic').topic_confidence.mean()}).reset_index()
priority['priority_score']=priority.volume*priority.negative_rate/100
display(priority.sort_values('priority_score',ascending=False).round(2))
plt.scatter(priority.volume,priority.negative_rate,s=priority.priority_score*5,alpha=.65,color='#E45756')
for _,r in priority.iterrows(): plt.annotate(f"Topic {int(r.dominant_topic)}",(r.volume,r.negative_rate),xytext=(4,4),textcoords='offset points')
plt.title('Priority Map: Volume vs Negative Sentiment'); plt.xlabel('Review Volume'); plt.ylabel('Negative Reviews (%)'); plt.tight_layout(); plt.show()

## 9. Interpretation, limitations and recommendations

### Interpretation
- Use macro F1, per-class recall and the confusion matrix to assess sentiment performance.
- Assign business labels to LDA topics only after inspecting top terms and representative reviews.
- Prioritise topics with both high volume and a high negative rate.

### Limitations
- Synthetic data cannot prove facts about a real medical-aid provider.
- Real online reviews are self-selected and may overrepresent extreme experiences.
- Sarcasm, multilingual text and domain terminology may reduce performance.
- LDA is a bag-of-words model and has limited contextual understanding.

### Recommended production controls
Validate topics with service specialists, manually audit low-confidence reviews, remove personal information, monitor drift monthly and retrain when macro F1 or topic coherence declines materially.

## 10. Export analysis-ready results

In [ ]:
df['predicted_sentiment']=best_model.predict(df.clean_text)
out=DATA_PATH.parent/'medical_aid_text_analysis_results.csv'
df[['review_id','review_date','provider','channel','province','rating','review_text','sentiment','predicted_sentiment','known_concern_category','dominant_topic','topic_confidence','word_count']].to_csv(out,index=False,encoding='utf-8-sig')
print('Saved:',out)